# First Improvement Notebook - Cross-Modal Transformer + Harmonization

This notebook includes the first improvement architechure which has:

- dual **3D MRI/PET encoders**
- **cross-modal Transformer**
- **prototype-distance OOD analyzer**
- **residual FiLM-style harmonization block**
- **single classification head**

## Protocol
This notebook supports the **same two-direction evaluation style as the baseline**:

- **ADNI1 → ADNI2**
- **ADNI2 → ADNI1**

For each direction:
- the **source** dataset is split into **train / val / test**
- the model trains on **source train**
- early stopping / selection uses **source val**
- the **OOD result** is reported on the **target test** split


In [ ]:

%pip -q install nibabel pydicom simpleitk monai scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 133.0 MB/s eta 0:00:00


In [ ]:
import os
import gc
import json
import time
import math
import random
import shutil
import warnings
from dataclasses import dataclass
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
)

import nibabel as nib
import pydicom
import SimpleITK as sitk

from monai.transforms import (
    Compose,
    RandGaussianNoise,
    RandGaussianSmooth,
    RandBiasField,
    RandAdjustContrast,
    RandAffine,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


DEVICE: cuda


## 1) Mount Drive and configure paths

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    print("Drive mount skipped:", e)
    IN_COLAB = False

Mounted at /content/drive


In [ ]:
@dataclass
class CFG:

    DRIVE_ROOT: str = "/content/drive/MyDrive/dl_project_baseline"
    WORK_ROOT: str = "/content/first_improvement_work"
    CACHE_ROOT: str = "/content/first_improvement_work/cache"
    CHECKPOINT_ROOT: str = "/content/first_improvement_work/checkpoints"

    FORCE_REEXTRACT: bool = False
    CLEAR_CACHE: bool = False

    MAX_PAIR_GAP_DAYS: int = 180
    PREFILTER_BEFORE_SPLIT: bool = True
    TEST_SIZE: float = 0.30
    VAL_FROM_TRAIN_RATIO: float = 0.30 / 0.70


    TARGET_SHAPE: tuple = (96, 112, 96)

    BATCH_SIZE: int = 2
    NUM_WORKERS: int = 0


    EMBED_DIM: int = 256
    TOKEN_GRID: tuple = (4, 4, 4)
    TRANSFORMER_HEADS: int = 4
    TRANSFORMER_LAYERS: int = 3
    DROPOUT: float = 0.10
    NUM_CLASSES: int = 2

    EPOCHS: int = 30
    LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    AMP: bool = True
    CONSISTENCY_WEIGHT: float = 0.5
    USE_IDENTITY_LOSS: bool = False
    IDENTITY_WEIGHT: float = 0.1
    EARLY_STOPPING_PATIENCE: int = 8

    RUN_BOTH_DIRECTIONS: bool = True
    RUN_DIRECTION: str = "ADNI2_TO_ADNI1"

cfg = CFG()

DRIVE_ROOT = Path(cfg.DRIVE_ROOT) if IN_COLAB else Path.cwd()
WORK_ROOT = Path(cfg.WORK_ROOT) if IN_COLAB else (Path.cwd() / "first_improvement_work")
CACHE_ROOT = Path(cfg.CACHE_ROOT) if IN_COLAB else (WORK_ROOT / "cache")
CHECKPOINT_ROOT = Path(cfg.CHECKPOINT_ROOT) if IN_COLAB else (WORK_ROOT / "checkpoints")

for p in [WORK_ROOT, CACHE_ROOT, CHECKPOINT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

if cfg.CLEAR_CACHE:
    shutil.rmtree(CACHE_ROOT, ignore_errors=True)
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT     :", DRIVE_ROOT)
print("WORK_ROOT      :", WORK_ROOT)
print("CACHE_ROOT     :", CACHE_ROOT)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)

DRIVE_ROOT     : /content/drive/MyDrive/dl_project_baseline
WORK_ROOT      : /content/first_improvement_work
CACHE_ROOT     : /content/first_improvement_work/cache
CHECKPOINT_ROOT: /content/first_improvement_work/checkpoints


In [ ]:
def pick_first_existing(root: Path, candidates):
    for name in candidates:
        p = root / name
        if p.exists():
            return p
    return None

FILE_CANDIDATES = {
    "ADNI1_MRI_CSV": [
        "baseline_exact_adni1_mri_4_15_2026.csv",
        "baseline_exact_adni1_mri_4_14_2026.csv",
    ],
    "ADNI1_PET_CSV": [
        "baseline_exact_adni1_pet_4_15_2026.csv",
        "baseline_exact_adni1_pet_4_14_2026.csv",
    ],
    "ADNI2_MRI_CSV": [
        "baseline_exact_adni2_mri_4_15_2026.csv",
        "baseline_exact_adni2_mri_4_14_2026.csv",
    ],
    "ADNI2_PET_CSV": [
        "baseline_exact_adni2_pet_4_15_2026.csv",
        "baseline_exact_adni2_pet_4_14_2026.csv",
    ],
    "ADNI1_MRI_ZIPS": [
        "baseline_exact_adni1_mri.zip",
    ],
    "ADNI1_PET_ZIPS": [
        "baseline_exact_adni1_pet.zip",
        "baseline_exact_adni1_pet_dataset.zip",
    ],
    "ADNI2_MRI_ZIPS": [
        "baseline_exact_adni2_mri.zip",
    ],
    "ADNI2_PET_ZIPS": [
        "baseline_exact_adni2_pet.zip",
    ],
}

CSV_FILES = {
    "ADNI1_MRI": pick_first_existing(DRIVE_ROOT, FILE_CANDIDATES["ADNI1_MRI_CSV"]),
    "ADNI1_PET": pick_first_existing(DRIVE_ROOT, FILE_CANDIDATES["ADNI1_PET_CSV"]),
    "ADNI2_MRI": pick_first_existing(DRIVE_ROOT, FILE_CANDIDATES["ADNI2_MRI_CSV"]),
    "ADNI2_PET": pick_first_existing(DRIVE_ROOT, FILE_CANDIDATES["ADNI2_PET_CSV"]),
}
ZIP_FILES = {
    "ADNI1_MRI": [p for p in [pick_first_existing(DRIVE_ROOT, [x]) for x in FILE_CANDIDATES["ADNI1_MRI_ZIPS"]] if p is not None],
    "ADNI1_PET": [p for p in [pick_first_existing(DRIVE_ROOT, [x]) for x in FILE_CANDIDATES["ADNI1_PET_ZIPS"]] if p is not None],
    "ADNI2_MRI": [p for p in [pick_first_existing(DRIVE_ROOT, [x]) for x in FILE_CANDIDATES["ADNI2_MRI_ZIPS"]] if p is not None],
    "ADNI2_PET": [p for p in [pick_first_existing(DRIVE_ROOT, [x]) for x in FILE_CANDIDATES["ADNI2_PET_ZIPS"]] if p is not None],
}

print("CSV_FILES")
for k, v in CSV_FILES.items():
    print(f"  {k}: {v}")

print("\nZIP_FILES")
for k, v in ZIP_FILES.items():
    print(f"  {k}: {[str(x) for x in v]}")

CSV_FILES
  ADNI1_MRI: /content/drive/MyDrive/dl_project_baseline/baseline_exact_adni1_mri_4_15_2026.csv
  ADNI1_PET: /content/drive/MyDrive/dl_project_baseline/baseline_exact_adni1_pet_4_15_2026.csv
  ADNI2_MRI: /content/drive/MyDrive/dl_project_baseline/baseline_exact_adni2_mri_4_15_2026.csv
  ADNI2_PET: /content/drive/MyDrive/dl_project_baseline/baseline_exact_adni2_pet_4_15_2026.csv

ZIP_FILES
  ADNI1_MRI: ['/content/drive/MyDrive/dl_project_baseline/baseline_exact_adni1_mri.zip']
  ADNI1_PET: ['/content/drive/MyDrive/dl_project_baseline/baseline_exact_adni1_pet.zip', '/content/drive/MyDrive/dl_project_baseline/baseline_exact_adni1_pet_dataset.zip']
  ADNI2_MRI: ['/content/drive/MyDrive/dl_project_baseline/baseline_exact_adni2_mri.zip']
  ADNI2_PET: ['/content/drive/MyDrive/dl_project_baseline/baseline_exact_adni2_pet.zip']


## 2) Extract archives

In [ ]:
import subprocess

def ensure_extracted(zip_path: Path, out_dir: Path):
    if zip_path is None or not zip_path.exists():
        print(f"[missing] {zip_path}")
        return None

    stamp = out_dir / ".done"

    if out_dir.exists() and stamp.exists() and not cfg.FORCE_REEXTRACT:
        print(f"[cached] {out_dir}")
        return out_dir

    if out_dir.exists() and cfg.FORCE_REEXTRACT:
        shutil.rmtree(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[extracting] {zip_path.name} -> {out_dir}")

    result = subprocess.run(
        ["unzip", "-q", "-o", str(zip_path), "-d", str(out_dir)],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("System unzip failed, falling back to Python zipfile...")
        import zipfile
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(out_dir)

    stamp.touch()
    return out_dir

EXTRACTED = {}
for key, paths in ZIP_FILES.items():
    EXTRACTED[key] = []
    for zpath in paths:
        out_dir = WORK_ROOT / "extracted" / zpath.stem
        extracted = ensure_extracted(zpath, out_dir)
        if extracted is not None:
            EXTRACTED[key].append(extracted)

print("\nExtracted roots:")
for k, v in EXTRACTED.items():
    print(k, "->", [str(x) for x in v])

[extracting] baseline_exact_adni1_mri.zip -> /content/first_improvement_work/extracted/baseline_exact_adni1_mri
[extracting] baseline_exact_adni1_pet.zip -> /content/first_improvement_work/extracted/baseline_exact_adni1_pet
[extracting] baseline_exact_adni1_pet_dataset.zip -> /content/first_improvement_work/extracted/baseline_exact_adni1_pet_dataset
[extracting] baseline_exact_adni2_mri.zip -> /content/first_improvement_work/extracted/baseline_exact_adni2_mri
[extracting] baseline_exact_adni2_pet.zip -> /content/first_improvement_work/extracted/baseline_exact_adni2_pet

Extracted roots:
ADNI1_MRI -> ['/content/first_improvement_work/extracted/baseline_exact_adni1_mri']
ADNI1_PET -> ['/content/first_improvement_work/extracted/baseline_exact_adni1_pet', '/content/first_improvement_work/extracted/baseline_exact_adni1_pet_dataset']
ADNI2_MRI -> ['/content/first_improvement_work/extracted/baseline_exact_adni2_mri']
ADNI2_PET -> ['/content/first_improvement_work/extracted/baseline_exact_adni

## 3) Load CSVs and build MRI/PET pairs

In [ ]:
def load_csv(path: Path, modality_name: str, dataset_name: str):
    if path is None or not path.exists():
        raise FileNotFoundError(f"Missing CSV: {path}")

    df = pd.read_csv(path).copy()
    rename_map = {
        "Image Data ID": "image_id",
        "Subject": "subject",
        "Group": "group",
        "Sex": "sex",
        "Age": "age",
        "Visit": "visit",
        "Modality": "modality",
        "Description": "description",
        "Type": "type",
        "Acq Date": "acq_date",
        "Format": "format",
        "Downloaded": "downloaded",
    }
    df = df.rename(columns=rename_map)
    df["dataset"] = dataset_name
    df["modality_name"] = modality_name
    df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")
    df["subject"] = df["subject"].astype(str)
    df["visit"] = df["visit"].astype(str)
    df["group"] = df["group"].astype(str).str.upper()
    df["label_num"] = df["group"].map({"CN": 0, "AD": 1})
    df = df[df["group"].isin(["CN", "AD"])].copy()
    return df

adni1_mri = load_csv(CSV_FILES["ADNI1_MRI"], "MRI", "ADNI1")
adni1_pet = load_csv(CSV_FILES["ADNI1_PET"], "PET", "ADNI1")
adni2_mri = load_csv(CSV_FILES["ADNI2_MRI"], "MRI", "ADNI2")
adni2_pet = load_csv(CSV_FILES["ADNI2_PET"], "PET", "ADNI2")

for name, df in [
    ("ADNI1 MRI", adni1_mri),
    ("ADNI1 PET", adni1_pet),
    ("ADNI2 MRI", adni2_mri),
    ("ADNI2 PET", adni2_pet),
]:
    print(f"\n{name}: {df.shape}")
    print(df[['subject', 'group', 'visit', 'image_id', 'acq_date']].head(3))
    print(df['group'].value_counts().to_dict())


ADNI1 MRI: (57, 15)
      subject group visit image_id   acq_date
0  127_S_0754    AD   m24  I119425 2008-10-03
1  127_S_0259    CN   m48  I168410 2010-03-19
2  126_S_0680    CN   m48  I185557 2010-07-21
{'CN': 32, 'AD': 25}

ADNI1 PET: (57, 15)
      subject group visit image_id   acq_date
0  127_S_0754    AD   m24  I124457 2008-10-03
1  127_S_0259    CN   m48  I170056 2010-03-19
2  126_S_0680    CN   m48  I187885 2010-07-29
{'CN': 32, 'AD': 25}

ADNI2 MRI: (264, 15)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v02  I276860 2012-01-06
1  941_S_4365    CN   v21  I418940 2014-04-03
2  941_S_4292    CN   v21  I391682 2013-09-23
{'CN': 174, 'AD': 90}

ADNI2 PET: (264, 15)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v03  I283469 2012-02-08
1  941_S_4365    CN   v21  I420334 2014-04-16
2  941_S_4292    CN   v21  I393814 2013-10-09
{'CN': 174, 'AD': 90}


In [ ]:
def build_nearest_pairs(mri_df, pet_df, dataset_name, max_gap_days=180):
    shared_subjects = sorted(set(mri_df["subject"]) & set(pet_df["subject"]))
    rows = []

    for subj in shared_subjects:
        msub = mri_df[mri_df["subject"] == subj].copy()
        psub = pet_df[pet_df["subject"] == subj].copy()

        best = None
        best_key = None

        for _, mr in msub.iterrows():
            for _, pr in psub.iterrows():
                same_group = int(mr["group"] == pr["group"])
                if pd.isna(mr["acq_date"]) or pd.isna(pr["acq_date"]):
                    gap_days = 999999
                else:
                    gap_days = abs((mr["acq_date"] - pr["acq_date"]).days)

                score = (-same_group, gap_days)
                if best is None or score < best_key:
                    best = (mr, pr)
                    best_key = score

        if best is None:
            continue

        mr, pr = best
        if mr["group"] != pr["group"]:
            continue
        if pd.isna(mr["acq_date"]) or pd.isna(pr["acq_date"]):
            continue

        gap_days = abs((mr["acq_date"] - pr["acq_date"]).days)
        if gap_days > max_gap_days:
            continue

        rows.append({
            "dataset": dataset_name,
            "subject": subj,
            "group": mr["group"],
            "label_num": int(mr["label_num"]),
            "sex": mr["sex"],
            "age": float(mr["age"]),
            "mri_visit": mr["visit"],
            "pet_visit": pr["visit"],
            "mri_image_id": str(mr["image_id"]),
            "pet_image_id": str(pr["image_id"]),
            "mri_date": mr["acq_date"],
            "pet_date": pr["acq_date"],
            "pair_gap_days": int(gap_days),
        })

    out = pd.DataFrame(rows).sort_values(["dataset", "subject"]).reset_index(drop=True)
    return out

paired_adni1 = build_nearest_pairs(adni1_mri, adni1_pet, "ADNI1", max_gap_days=cfg.MAX_PAIR_GAP_DAYS)
paired_adni2 = build_nearest_pairs(adni2_mri, adni2_pet, "ADNI2", max_gap_days=cfg.MAX_PAIR_GAP_DAYS)

print("paired_adni1:", paired_adni1.shape, paired_adni1["group"].value_counts().to_dict())
print("paired_adni2:", paired_adni2.shape, paired_adni2["group"].value_counts().to_dict())
display(paired_adni1.head())
display(paired_adni2.head())

paired_adni1: (57, 13) {'CN': 32, 'AD': 25}
paired_adni2: (264, 13) {'CN': 174, 'AD': 90}


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI1,005_S_0221,AD,1,M,68.0,m06,m06,I25942,I25918,2006-10-06,2006-10-06,0
1,ADNI1,005_S_0223,CN,0,F,79.0,m06,m06,I26116,I26108,2006-10-11,2006-10-11,0
2,ADNI1,005_S_0610,CN,0,M,80.0,m06,m06,I39068,I39087,2007-02-09,2007-02-09,0
3,ADNI1,005_S_0929,AD,1,M,83.0,m06,m06,I53858,I57389,2007-05-09,2007-06-19,41
4,ADNI1,005_S_1341,AD,1,F,72.0,m06,m06,I76567,I76770,2007-10-02,2007-10-02,0


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI2,002_S_0295,CN,0,M,90.0,v06,v06,I238627,I239487,2011-06-02,2011-06-09,7
1,ADNI2,002_S_0413,CN,0,F,82.0,v06,v06,I240812,I240813,2011-06-16,2011-06-17,1
2,ADNI2,002_S_0685,CN,0,F,96.0,v11,v11,I322437,I321228,2012-07-27,2012-08-02,6
3,ADNI2,002_S_1261,CN,0,F,77.0,v11,v11,I361610,I363184,2013-02-27,2013-03-07,8
4,ADNI2,002_S_1280,CN,0,F,77.0,v11,v11,I361326,I360640,2013-02-26,2013-02-19,7


## 4) Build path indexes and robust volume loaders

In [ ]:
VOLUME_EXTENSIONS = (
    ".nii", ".nii.gz", ".img", ".hdr", ".v", ".mnc", ".nrrd", ".mha", ".mhd", ".mgz"
)

def build_path_index(search_roots):
    index = {"dirs": [], "vol_files": []}

    for root in search_roots:
        if root is None or not Path(root).exists():
            continue

        for p in Path(root).rglob("*"):
            low = str(p).lower()
            if p.is_dir():
                index["dirs"].append((low, p))
            elif p.is_file():
                if low.endswith(VOLUME_EXTENSIONS):
                    index["vol_files"].append((low, p))
    return index

INDEXES = {
    "ADNI1_MRI": build_path_index(EXTRACTED["ADNI1_MRI"]),
    "ADNI1_PET": build_path_index(EXTRACTED["ADNI1_PET"]),
    "ADNI2_MRI": build_path_index(EXTRACTED["ADNI2_MRI"]),
    "ADNI2_PET": build_path_index(EXTRACTED["ADNI2_PET"]),
}

for k, idx in INDEXES.items():
    print(k, "dirs:", len(idx["dirs"]), "| vol files:", len(idx["vol_files"]))


def locate_study_path(index, image_id, subject, visit=None):
    image_id = str(image_id).lower()
    subject = str(subject).lower()
    visit = None if visit is None else str(visit).lower()

    def is_v_file(p):
        return str(p).lower().endswith(".v")

    hit_dirs = [p for low, p in index["dirs"] if image_id in low]
    if hit_dirs:
        return sorted(hit_dirs, key=lambda x: len(str(x)), reverse=True)[0]

    hit_files = [p for low, p in index["vol_files"] if image_id in low and not is_v_file(p)]
    if hit_files:
        return sorted(hit_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    hit_v_files = [p for low, p in index["vol_files"] if image_id in low and is_v_file(p)]
    if hit_v_files:
        return sorted(hit_v_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    subject_dirs = []
    for low, p in index["dirs"]:
        if subject in low and (visit is None or visit in low):
            subject_dirs.append(p)
    if subject_dirs:
        return sorted(subject_dirs, key=lambda x: len(str(x)), reverse=True)[0]

    subject_dirs = [p for low, p in index["dirs"] if subject in low]
    if subject_dirs:
        return sorted(subject_dirs, key=lambda x: len(str(x)), reverse=True)[0]

    subject_files = []
    for low, p in index["vol_files"]:
        if subject in low and (visit is None or visit in low) and not is_v_file(p):
            subject_files.append(p)
    if subject_files:
        return sorted(subject_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    subject_v_files = []
    for low, p in index["vol_files"]:
        if subject in low and (visit is None or visit in low) and is_v_file(p):
            subject_v_files.append(p)
    if subject_v_files:
        return sorted(subject_v_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    raise FileNotFoundError(
        f"Could not locate study path for image_id={image_id}, subject={subject}, visit={visit}"
    )

ADNI1_MRI dirs: 229 | vol files: 0
ADNI1_PET dirs: 230 | vol files: 48
ADNI2_MRI dirs: 1053 | vol files: 0
ADNI2_PET dirs: 1053 | vol files: 173


In [ ]:
PAPER_PREPROCESSED_KEYWORDS = (
    "cat12", "spm", "mni", "smwc", "mwp", "warped", "normalized", "normalised",
    "registered", "coreg", "brain", "strip", "skull", "resliced", "resample",
    "smooth", "wc1", "wc2", "mrireg", "petreg"
)

def _is_probably_preprocessed_path(path: Path):
    low = str(path).lower()
    return any(k in low for k in PAPER_PREPROCESSED_KEYWORDS)

def _ensure_3d_volume(arr, source):
    arr = np.asarray(arr, dtype=np.float32)
    while arr.ndim > 3:
        arr = arr[..., 0]
    if arr.ndim != 3:
        raise RuntimeError(f"Expected 3D volume from {source}, got shape {arr.shape}")
    return arr.astype(np.float32)

def _load_analyze_like_hdr_pair(hdr_path: Path):
    hdr_path = Path(hdr_path)
    low = str(hdr_path).lower()
    if not low.endswith(".hdr"):
        raise RuntimeError(f"Not a .hdr file: {hdr_path}")

    candidates = [hdr_path.with_suffix(".img")]
    if low.endswith(".i.hdr"):
        candidates.append(Path(str(hdr_path)[:-4]))
    candidates.append(Path(str(hdr_path)[:-4]))

    seen = set()
    unique_candidates = []
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            unique_candidates.append(c)

    data_path = None
    for c in unique_candidates:
        if c.exists() and c.is_file():
            data_path = c
            break
    if data_path is None:
        raise RuntimeError(f"Could not find paired binary file for header: {hdr_path}")

    try:
        from nibabel.analyze import AnalyzeHeader
        with open(hdr_path, "rb") as f_hdr:
            hdr = AnalyzeHeader.from_fileobj(f_hdr)
        with open(data_path, "rb") as f_img:
            arr = hdr.data_from_fileobj(f_img)
        return _ensure_3d_volume(arr, f"{hdr_path} + {data_path}")
    except Exception as e:
        raise RuntimeError(f"Failed to load Analyze-like pair: {hdr_path} + {data_path}") from e

def load_single_volume_file(path: Path):
    path = Path(path)
    low = str(path).lower()

    if low.endswith(".v"):
        try:
            from nibabel import ecat
            img = ecat.load(str(path))
            arr = np.asarray(img.get_fdata(), dtype=np.float32)
            return _ensure_3d_volume(arr, path)
        except Exception:
            pass

    if low.endswith(".hdr"):
        try:
            return _load_analyze_like_hdr_pair(path)
        except Exception:
            pass

    try:
        img = nib.load(str(path))
        arr = np.asarray(img.get_fdata(), dtype=np.float32)
        return _ensure_3d_volume(arr, path)
    except Exception:
        pass

    try:
        img = sitk.ReadImage(str(path))
        arr = sitk.GetArrayFromImage(img).astype(np.float32)
        arr = np.transpose(arr, (2, 1, 0))
        return _ensure_3d_volume(arr, path)
    except Exception as e:
        raise RuntimeError(f"Could not load single-volume file: {path}") from e

def _dicom_sort_value(ds):
    instance_no = getattr(ds, "InstanceNumber", None)
    if instance_no is not None:
        try:
            return float(instance_no)
        except Exception:
            pass

    ipp = getattr(ds, "ImagePositionPatient", None)
    if ipp is not None and len(ipp) >= 3:
        try:
            return float(ipp[-1])
        except Exception:
            pass

    slice_location = getattr(ds, "SliceLocation", None)
    if slice_location is not None:
        try:
            return float(slice_location)
        except Exception:
            pass

    return 0.0

def load_dicom_series_from_dir(series_dir: Path):
    series_dir = Path(series_dir)
    try:
        reader = sitk.ImageSeriesReader()
        series_ids = reader.GetGDCMSeriesIDs(str(series_dir))
        if series_ids:
            series_candidates = []
            for sid in series_ids:
                files = list(reader.GetGDCMSeriesFileNames(str(series_dir), sid))
                series_candidates.append((len(files), sid, files))
            for _, sid, files in sorted(series_candidates, reverse=True):
                try:
                    reader.SetFileNames(files)
                    img = reader.Execute()
                    arr = sitk.GetArrayFromImage(img).astype(np.float32)
                    arr = np.transpose(arr, (2, 1, 0))
                    return _ensure_3d_volume(arr, f"{series_dir}::{sid}")
                except Exception:
                    continue
    except Exception:
        pass

    grouped_slices = {}
    multiframe_volumes = []

    for p in series_dir.rglob("*"):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=False, force=True)
            if not hasattr(ds, "pixel_array"):
                continue
            arr = np.asarray(ds.pixel_array, dtype=np.float32)
        except Exception:
            continue

        series_uid = str(getattr(ds, "SeriesInstanceUID", "unknown"))
        if arr.ndim == 3:
            vol = np.transpose(arr, (1, 2, 0))
            multiframe_volumes.append((arr.shape[-1], vol, p))
        elif arr.ndim == 2:
            grouped_slices.setdefault(series_uid, []).append((_dicom_sort_value(ds), arr))

    if multiframe_volumes:
        _, vol, source_path = sorted(multiframe_volumes, reverse=True)[0]
        return _ensure_3d_volume(vol, source_path)

    if grouped_slices:
        best_uid = max(grouped_slices.keys(), key=lambda uid: len(grouped_slices[uid]))
        slices = sorted(grouped_slices[best_uid], key=lambda x: x[0])
        vol = np.stack([sl for _, sl in slices], axis=-1)
        return _ensure_3d_volume(vol, f"{series_dir}::{best_uid}")

    raise RuntimeError(f"Could not build DICOM volume from directory: {series_dir}")

def load_volume(path):
    path = Path(path)
    if path.is_file():
        return load_single_volume_file(path)
    if path.is_dir():
        volume_files = []
        for p in path.rglob("*"):
            if p.is_file():
                low = str(p).lower()
                if low.endswith(VOLUME_EXTENSIONS):
                    volume_files.append(p)
        volume_files = sorted(volume_files)
        preferred_files = [
            p for p in volume_files
            if _is_probably_preprocessed_path(p)
        ]
        if preferred_files:
            best = sorted(
                preferred_files,
                key=lambda x: x.stat().st_size if x.exists() else 0,
                reverse=True
            )[0]
            return load_single_volume_file(best)
        if volume_files:
            best = sorted(
                volume_files,
                key=lambda x: x.stat().st_size if x.exists() else 0,
                reverse=True
            )[0]
            return load_single_volume_file(best)
        return load_dicom_series_from_dir(path)
    raise RuntimeError(f"Path is neither file nor directory: {path}")

In [ ]:
def _ensure_3d_float(volume):
    v = np.asarray(volume, dtype=np.float32)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    while v.ndim > 3:
        v = v[..., 0]
    if v.ndim != 3:
        raise ValueError(f"Expected 3D volume after squeezing, got shape {v.shape}")
    return v.astype(np.float32)

def robust_clip(volume, lower_q=0.5, upper_q=99.5):
    v = np.asarray(volume, dtype=np.float32)
    finite_mask = np.isfinite(v)
    if finite_mask.sum() == 0:
        return np.zeros_like(v, dtype=np.float32)
    lo, hi = np.percentile(v[finite_mask], [lower_q, upper_q])
    v = np.clip(v, lo, hi)
    return v.astype(np.float32)

def crop_to_foreground(volume, modality, margin=4):
    v = _ensure_3d_float(volume)
    nonzero = np.abs(v) > 0
    foreground_values = v[nonzero] if nonzero.any() else v[np.isfinite(v)]
    if foreground_values.size == 0:
        return v

    threshold = np.percentile(foreground_values, 35 if modality.upper() == "MRI" else 55)
    mask = (v > threshold) | nonzero
    coords = np.argwhere(mask)
    if coords.size == 0:
        return v

    lo = np.maximum(coords.min(axis=0) - margin, 0)
    hi = np.minimum(coords.max(axis=0) + margin + 1, np.array(v.shape))
    cropped = v[int(lo[0]):int(hi[0]), int(lo[1]):int(hi[1]), int(lo[2]):int(hi[2])]
    return cropped.astype(np.float32) if cropped.size > 0 else v

def maybe_n4_bias_correct(volume):
    v = _ensure_3d_float(volume)
    try:
        itk = sitk.GetImageFromArray(np.transpose(v, (2, 1, 0)))
        mask = sitk.OtsuThreshold(itk, 0, 1, 128)
        corrected = sitk.N4BiasFieldCorrection(itk, mask)
        out = sitk.GetArrayFromImage(corrected).astype(np.float32)
        return np.transpose(out, (2, 1, 0))
    except Exception:
        return v

def maybe_gaussian_smooth(volume, sigma=1.0):
    v = _ensure_3d_float(volume)
    try:
        itk = sitk.GetImageFromArray(np.transpose(v, (2, 1, 0)))
        smoothed = sitk.SmoothingRecursiveGaussian(itk, sigma=float(sigma))
        out = sitk.GetArrayFromImage(smoothed).astype(np.float32)
        return np.transpose(out, (2, 1, 0))
    except Exception:
        return v

def normalize_mri(volume):
    v = robust_clip(volume)
    mask = v > np.percentile(v, 20)
    mean = v[mask].mean() if mask.sum() > 0 else v.mean()
    std = v[mask].std() + 1e-6 if mask.sum() > 0 else v.std() + 1e-6
    v = (v - mean) / std
    return v.astype(np.float32)

def normalize_pet(volume):
    v = robust_clip(volume)
    v = v - v.min()
    v = v / (v.max() + 1e-6)
    return v.astype(np.float32)

def resize_volume(volume, target_shape=None):
    target_shape = cfg.TARGET_SHAPE if target_shape is None else target_shape
    x = torch.tensor(volume, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    x = F.interpolate(x, size=target_shape, mode="trilinear", align_corners=False)
    return x.squeeze(0).squeeze(0).cpu().numpy().astype(np.float32)

def preprocess_volume(volume, modality, source_path=None):
    v = _ensure_3d_float(volume)
    source_path = None if source_path is None else Path(source_path)
    paper_like_input = (
        source_path is not None and _is_probably_preprocessed_path(source_path)
    )

    if not paper_like_input:
        v = crop_to_foreground(v, modality=modality, margin=4)

    if modality.upper() == "MRI":
        if not paper_like_input:
            v = maybe_n4_bias_correct(v)
            v = maybe_gaussian_smooth(v, sigma=1.0)
        v = normalize_mri(v)
    elif modality.upper() == "PET":
        if not paper_like_input:
            v = maybe_gaussian_smooth(v, sigma=1.0)
        v = normalize_pet(v)
    else:
        raise ValueError(f"Unknown modality: {modality}")

    if tuple(v.shape) != tuple(cfg.TARGET_SHAPE):
        v = resize_volume(v, target_shape=cfg.TARGET_SHAPE)

    if modality.upper() == "MRI":
        v = normalize_mri(v)
    else:
        v = normalize_pet(v)
    return v.astype(np.float32)

def cache_key(row, modality):
    image_id = row["mri_image_id"] if modality == "MRI" else row["pet_image_id"]
    return f"{row['dataset']}__{row['subject']}__{modality}__{image_id}.npy"

def get_cached_or_build(row, modality, index_key):
    out_path = CACHE_ROOT / cache_key(row, modality)
    if out_path.exists():
        try:
            arr = np.load(out_path)
            return arr.astype(np.float32)
        except Exception:
            try:
                out_path.unlink()
            except Exception:
                pass

    if modality == "MRI":
        image_id = row["mri_image_id"]
        visit = row["mri_visit"]
    else:
        image_id = row["pet_image_id"]
        visit = row["pet_visit"]

    index = INDEXES[index_key]

    try:
        source_path = locate_study_path(
            index,
            image_id=image_id,
            subject=row["subject"],
            visit=visit,
        )
        volume = load_volume(source_path)
        volume = preprocess_volume(volume, modality=modality, source_path=source_path)
        volume = volume.astype(np.float32)
        np.save(out_path, volume)
        return volume
    except Exception:
        try:
            if out_path.exists():
                out_path.unlink()
        except Exception:
            pass
        raise

In [ ]:
def filter_loadable_pairs(df, split_name):
    keep_indices = []
    dropped = []

    for idx in range(len(df)):
        row = df.iloc[idx].to_dict()
        dataset = row["dataset"]
        mri_index_key = f"{dataset}_MRI"
        pet_index_key = f"{dataset}_PET"
        try:
            _ = get_cached_or_build(row, modality="MRI", index_key=mri_index_key)
            _ = get_cached_or_build(row, modality="PET", index_key=pet_index_key)
            keep_indices.append(idx)
        except Exception as e:
            dropped.append({
                "split": split_name,
                "idx": idx,
                "dataset": row["dataset"],
                "subject": row["subject"],
                "label_num": row["label_num"],
                "mri_image_id": row["mri_image_id"],
                "pet_image_id": row["pet_image_id"],
                "error": str(e),
            })
            print(
                f"[drop] {split_name} | idx={idx} | dataset={row['dataset']} | "
                f"subject={row['subject']} | reason={str(e)[:180]}"
            )
            for modality in ["MRI", "PET"]:
                p = CACHE_ROOT / cache_key(row, modality)
                try:
                    if p.exists():
                        p.unlink()
                except Exception:
                    pass

    filtered_df = df.iloc[keep_indices].reset_index(drop=True)
    dropped_df = pd.DataFrame(dropped)
    print(f"{split_name}: kept {len(filtered_df)} / {len(df)}")
    if len(dropped_df) > 0:
        print(f"{split_name}: dropped {len(dropped_df)} unreadable pairs")
    else:
        print(f"{split_name}: dropped 0 unreadable pairs")
    return filtered_df, dropped_df

if cfg.PREFILTER_BEFORE_SPLIT:
    paired_adni1_f, paired_adni1_bad = filter_loadable_pairs(paired_adni1, "paired_adni1")
    paired_adni2_f, paired_adni2_bad = filter_loadable_pairs(paired_adni2, "paired_adni2")
else:
    paired_adni1_f, paired_adni1_bad = paired_adni1.copy(), pd.DataFrame()
    paired_adni2_f, paired_adni2_bad = paired_adni2.copy(), pd.DataFrame()

paired_df = pd.concat([paired_adni1_f, paired_adni2_f], ignore_index=True)
paired_csv_path = WORK_ROOT / "paired_first_improvement_table.csv"
paired_df.to_csv(paired_csv_path, index=False)
print("Saved paired table:", paired_csv_path)
print("paired_adni1_f:", paired_adni1_f.shape, paired_adni1_f["group"].value_counts().to_dict())
print("paired_adni2_f:", paired_adni2_f.shape, paired_adni2_f["group"].value_counts().to_dict())

paired_adni1: kept 57 / 57
paired_adni1: dropped 0 unreadable pairs


sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
ERROR:nibabel.global:data code 24951 not recognized; not attempting fix


[drop] paired_adni2 | idx=101 | dataset=ADNI2 | subject=032_S_0479 | reason=Could not load single-volume file: /content/first_improvement_work/extracted/baseline_exact_adni2_pet/ADNI/032_S_0479/ADNI_Brain_PET__Raw_FDG/2011-09-26_15_05_42.0/I259489/ADNI_032


sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
ERROR:nibabel.global:data code 24951 not recognized; not attempting fix


[drop] paired_adni2 | idx=102 | dataset=ADNI2 | subject=032_S_0677 | reason=Could not load single-volume file: /content/first_improvement_work/extracted/baseline_exact_adni2_pet/ADNI/032_S_0677/ADNI_Brain_PET__Raw_FDG/2011-09-26_12_46_04.0/I259815/ADNI_032


...

[drop] paired_adni2 | idx=208 | dataset=ADNI2 | subject=128_S_4774 | reason=Could not load single-volume file: /content/first_improvement_work/extracted/baseline_exact_adni2_pet/ADNI/128_S_4774/ADNI_Brain_PET__Raw_FDG/2012-08-01_11_47_48.0/I321575/ADNI_128


sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
ERROR:nibabel.global:data code 24951 not recognized; not attempting fix


[drop] paired_adni2 | idx=209 | dataset=ADNI2 | subject=128_S_4792 | reason=Could not load single-volume file: /content/first_improvement_work/extracted/baseline_exact_adni2_pet/ADNI/128_S_4792/ADNI_Brain_PET__Raw_FDG/2012-07-25_11_00_56.0/I318819/ADNI_128


sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
ERROR:nibabel.global:data code 24951 not recognized; not attempting fix


[drop] paired_adni2 | idx=210 | dataset=ADNI2 | subject=128_S_4832 | reason=Could not load single-volume file: /content/first_improvement_work/extracted/baseline_exact_adni2_pet/ADNI/128_S_4832/ADNI_Brain_PET__Raw_FDG/2012-08-09_11_21_00.0/I323149/ADNI_128
paired_adni2: kept 243 / 264
paired_adni2: dropped 21 unreadable pairs
Saved paired table: /content/first_improvement_work/paired_first_improvement_table.csv
paired_adni1_f: (57, 13) {'CN': 32, 'AD': 25}
paired_adni2_f: (243, 13) {'CN': 160, 'AD': 83}


## 5) Create the per-domain splits, then run both directions

In [ ]:
def make_domain_splits(df, seed=42):
    df = df.reset_index(drop=True).copy()
    train_val_df, test_df = train_test_split(
        df,
        test_size=cfg.TEST_SIZE,
        random_state=seed,
        stratify=df["label_num"],
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=cfg.VAL_FROM_TRAIN_RATIO,
        random_state=seed,
        stratify=train_val_df["label_num"],
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

adni1_train, adni1_val, adni1_test = make_domain_splits(paired_adni1_f, seed=SEED)
adni2_train, adni2_val, adni2_test = make_domain_splits(paired_adni2_f, seed=SEED)

def show_df(name, dfx):
    print(name, dfx.shape, dfx["group"].value_counts().to_dict())

print("ADNI1 splits:")
for name, dfx in [("train", adni1_train), ("val", adni1_val), ("test", adni1_test)]:
    show_df(name, dfx)

print("\nADNI2 splits:")
for name, dfx in [("train", adni2_train), ("val", adni2_val), ("test", adni2_test)]:
    show_df(name, dfx)

ADNI1 splits:
train (22, 13) {'CN': 12, 'AD': 10}
val (17, 13) {'CN': 10, 'AD': 7}
test (18, 13) {'CN': 10, 'AD': 8}

ADNI2 splits:
train (97, 13) {'CN': 64, 'AD': 33}
val (73, 13) {'CN': 48, 'AD': 25}
test (73, 13) {'CN': 48, 'AD': 25}


## 6) Dataset + augmentations

In [ ]:
strong_view_transform = Compose([
    RandGaussianNoise(prob=0.30, mean=0.0, std=0.03),
    RandGaussianSmooth(prob=0.20, sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),
    RandBiasField(prob=0.20, coeff_range=(0.0, 0.1)),
    RandAdjustContrast(prob=0.20, gamma=(0.8, 1.2)),
    RandAffine(prob=0.25, rotate_range=(0.08, 0.08, 0.08), scale_range=(0.05, 0.05, 0.05), mode="bilinear"),
])

class PairedADNIDataset(Dataset):
    def __init__(self, df, return_augmented=False):
        self.df = df.reset_index(drop=True).copy()
        self.return_augmented = return_augmented

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()
        dataset = row["dataset"]
        mri_index_key = f"{dataset}_MRI"
        pet_index_key = f"{dataset}_PET"

        mri = get_cached_or_build(row, modality="MRI", index_key=mri_index_key)
        pet = get_cached_or_build(row, modality="PET", index_key=pet_index_key)

        mri = torch.tensor(mri, dtype=torch.float32).unsqueeze(0)
        pet = torch.tensor(pet, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(row["label_num"]), dtype=torch.long)

        item = {
            "mri": mri,
            "pet": pet,
            "label": label,
            "subject": row["subject"],
            "dataset": row["dataset"],
        }

        if self.return_augmented:
            item["mri_aug"] = strong_view_transform(mri.clone())
            item["pet_aug"] = strong_view_transform(pet.clone())
        else:
            item["mri_aug"] = mri.clone()
            item["pet_aug"] = pet.clone()
        return item

def make_loader(df, split_name, shuffle, return_augmented):
    ds = PairedADNIDataset(df, return_augmented=return_augmented)
    return DataLoader(
        ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

## 7) First-improvement model

In [ ]:

class ConvNormAct3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, norm="group"):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm = nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch) if norm == "group" else nn.InstanceNorm3d(out_ch)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block1 = ConvNormAct3D(in_ch, out_ch, stride=stride)
        self.block2 = nn.Sequential(
            nn.Conv3d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch),
        )
        self.act = nn.GELU()
        self.skip = nn.Identity()
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(num_groups=min(8, out_ch), num_channels=out_ch),
            )

    def forward(self, x):
        identity = self.skip(x)
        out = self.block1(x)
        out = self.block2(out)
        return self.act(out + identity)

class Encoder3D(nn.Module):
    def __init__(self, in_ch=1, base=32, out_ch=256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(in_ch, base, kernel_size=5, stride=2, padding=2, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=base),
            nn.GELU(),
        )
        self.layer1 = ResidualBlock3D(base, base, stride=1)
        self.layer2 = ResidualBlock3D(base, base * 2, stride=2)
        self.layer3 = ResidualBlock3D(base * 2, base * 4, stride=2)
        self.layer4 = ResidualBlock3D(base * 4, out_ch, stride=2)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x 

class Tokenizer3D(nn.Module):
    def __init__(self, in_ch, embed_dim, token_grid=(4,4,4)):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(token_grid)
        self.proj = nn.Conv3d(in_ch, embed_dim, kernel_size=1)

    def forward(self, x):
        x = self.pool(x)
        x = self.proj(x)
        b, c, d, h, w = x.shape
        x = x.flatten(2).transpose(1, 2) 
        return x

class CrossModalTransformer(nn.Module):
    def __init__(self, embed_dim=256, n_heads=4, n_layers=3, dropout=0.1, token_grid=(4,4,4)):
        super().__init__()
        num_tokens_per_modality = int(np.prod(token_grid))
        self.total_tokens = 1 + 2 * num_tokens_per_modality

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.total_tokens, embed_dim))
        self.modality_embed = nn.Parameter(torch.zeros(1, self.total_tokens, embed_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.modality_embed, std=0.02)

        self.num_tokens_per_modality = num_tokens_per_modality

    def forward(self, mri_tokens, pet_tokens):
        b = mri_tokens.size(0)
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, mri_tokens, pet_tokens], dim=1)

        modality_embed = self.modality_embed[:, :x.size(1), :].clone()
        modality_embed[:, 1:1+self.num_tokens_per_modality, :] += 0.5
        modality_embed[:, 1+self.num_tokens_per_modality:, :] -= 0.5

        x = x + self.pos_embed[:, :x.size(1), :] + modality_embed
        x = self.encoder(x)
        x = self.norm(x)

        cls_out = x[:, 0]
        return cls_out, x

class PrototypeBank(nn.Module):
    def __init__(self, num_classes, dim, momentum=0.9):
        super().__init__()
        self.num_classes = num_classes
        self.dim = dim
        self.momentum = momentum
        self.register_buffer("prototypes", torch.zeros(num_classes, dim))
        self.register_buffer("counts", torch.zeros(num_classes))

    @torch.no_grad()
    def update(self, embeddings, labels):
        for c in range(self.num_classes):
            mask = (labels == c)
            if mask.any():
                batch_proto = embeddings[mask].mean(dim=0)
                if self.counts[c] == 0:
                    self.prototypes[c] = batch_proto
                else:
                    self.prototypes[c] = self.momentum * self.prototypes[c] + (1 - self.momentum) * batch_proto
                self.counts[c] += mask.sum()

    def min_distance(self, embeddings):
        valid = self.counts > 0
        if valid.sum() == 0:
            return torch.zeros(embeddings.size(0), 1, device=embeddings.device)
        protos = self.prototypes[valid] 
        dists = torch.cdist(embeddings, protos, p=2)  
        min_d = dists.min(dim=1).values.unsqueeze(-1)
        return min_d

class HarmonizationBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.cond_mlp = nn.Sequential(
            nn.Linear(dim + 1, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim * 2),
        )
        self.delta_mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.gate_mlp = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, z, d_proto):
        z_n = self.norm(z)
        u = torch.cat([z_n, d_proto], dim=-1)
        gamma, beta = self.cond_mlp(u).chunk(2, dim=-1)
        z_film = gamma * z_n + beta
        delta = self.delta_mlp(z_film)
        alpha = self.gate_mlp(d_proto)
        z_corr = z + alpha * delta
        return z_corr, alpha, delta

class ClassifierHead(nn.Module):
    def __init__(self, dim, num_classes, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class FirstImprovementModel(nn.Module):
    def __init__(self, num_classes=2, embed_dim=256, token_grid=(4,4,4), n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.mri_encoder = Encoder3D(in_ch=1, base=32, out_ch=embed_dim)
        self.pet_encoder = Encoder3D(in_ch=1, base=32, out_ch=embed_dim)
        self.mri_tokenizer = Tokenizer3D(embed_dim, embed_dim, token_grid=token_grid)
        self.pet_tokenizer = Tokenizer3D(embed_dim, embed_dim, token_grid=token_grid)
        self.fusion = CrossModalTransformer(embed_dim=embed_dim, n_heads=n_heads, n_layers=n_layers, dropout=dropout, token_grid=token_grid)
        self.prototype_bank = PrototypeBank(num_classes=num_classes, dim=embed_dim, momentum=0.9)
        self.harmonizer = HarmonizationBlock(dim=embed_dim, dropout=dropout)
        self.classifier = ClassifierHead(dim=embed_dim, num_classes=num_classes, dropout=dropout)

    def encode(self, mri, pet):
        mri_feat = self.mri_encoder(mri)
        pet_feat = self.pet_encoder(pet)
        mri_tokens = self.mri_tokenizer(mri_feat)
        pet_tokens = self.pet_tokenizer(pet_feat)
        z, token_out = self.fusion(mri_tokens, pet_tokens)
        return z, token_out

    def forward(self, mri, pet):
        z, token_out = self.encode(mri, pet)
        d_proto = self.prototype_bank.min_distance(z)
        z_corr, alpha, delta = self.harmonizer(z, d_proto)
        logits = self.classifier(z_corr)
        return {
            "z": z,
            "z_corr": z_corr,
            "prototype_distance": d_proto,
            "alpha": alpha,
            "delta": delta,
            "logits": logits,
            "token_out": token_out,
        }

## 8) Losses, metrics, and training

In [ ]:
def compute_losses(model_out_clean, labels, model_out_aug=None):
    loss_cls = F.cross_entropy(model_out_clean["logits"], labels)

    loss_cons = torch.tensor(0.0, device=labels.device)
    if model_out_aug is not None:
        z1 = F.normalize(model_out_clean["z_corr"], dim=-1)
        z2 = F.normalize(model_out_aug["z_corr"], dim=-1)
        loss_cons = F.mse_loss(z1, z2)

    total = loss_cls + cfg.CONSISTENCY_WEIGHT * loss_cons

    loss_id = torch.tensor(0.0, device=labels.device)
    if cfg.USE_IDENTITY_LOSS:
        loss_id = F.mse_loss(model_out_clean["z_corr"], model_out_clean["z"])
        total = total + cfg.IDENTITY_WEIGHT * loss_id

    return {
        "total": total,
        "cls": loss_cls.detach(),
        "cons": loss_cons.detach(),
        "id": loss_id.detach(),
    }

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spe = tn / (tn + fp + 1e-8)
    sen = tp / (tp + fn + 1e-8)

    return {
        "ACC": acc * 100.0,
        "BACC": bacc * 100.0,
        "SEN": sen * 100.0,
        "SPE": spe * 100.0,
        "AUC": auc * 100.0 if not np.isnan(auc) else np.nan,
        "F1": f1 * 100.0,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

def validation_score(metrics):
    if not np.isnan(metrics.get("BACC", np.nan)):
        return float(metrics["BACC"])
    if not np.isnan(metrics.get("AUC", np.nan)):
        return float(metrics["AUC"])
    return float(metrics.get("ACC", 0.0))

def run_epoch(model, loader, optimizer=None, scaler=None, train=True):
    model.train(train)
    all_probs, all_labels = [], []
    total_loss = 0.0
    total_cls = 0.0
    total_cons = 0.0
    total_id = 0.0
    n_batches = 0

    for batch in tqdm(loader, leave=False):
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        mri_aug = batch["mri_aug"].to(DEVICE, non_blocking=True)
        pet_aug = batch["pet_aug"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.AMP and DEVICE.type == "cuda")):
            out_clean = model(mri, pet)
            out_aug = model(mri_aug, pet_aug) if train else None
            losses = compute_losses(out_clean, labels, model_out_aug=out_aug if train else None)
            loss = losses["total"]

        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            with torch.no_grad():
                model.prototype_bank.update(out_clean["z"].detach(), labels.detach())

        probs = torch.softmax(out_clean["logits"], dim=-1)[:, 1].detach().cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.detach().cpu().numpy())

        total_loss += float(loss.detach().cpu().item())
        total_cls += float(losses["cls"].cpu().item())
        total_cons += float(losses["cons"].cpu().item())
        total_id += float(losses["id"].cpu().item())
        n_batches += 1

    y_prob = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_labels, axis=0)

    metrics = compute_metrics(y_true, y_prob)
    metrics.update({
        "loss": total_loss / max(n_batches, 1),
        "loss_cls": total_cls / max(n_batches, 1),
        "loss_cons": total_cons / max(n_batches, 1),
        "loss_id": total_id / max(n_batches, 1),
    })
    return metrics

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    probs_all, labels_all = [], []
    proto_all, alpha_all = [], []
    for batch in loader:
        mri = batch["mri"].to(DEVICE)
        pet = batch["pet"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        out = model(mri, pet)
        probs = torch.softmax(out["logits"], dim=-1)[:, 1]
        probs_all.append(probs.cpu().numpy())
        labels_all.append(labels.cpu().numpy())
        proto_all.append(out["prototype_distance"].cpu().numpy())
        alpha_all.append(out["alpha"].cpu().numpy())
    return (
        np.concatenate(probs_all, axis=0),
        np.concatenate(labels_all, axis=0),
        np.concatenate(proto_all, axis=0),
        np.concatenate(alpha_all, axis=0),
    )

In [ ]:
def run_direction(source_name, target_name, source_train_df, source_val_df, target_test_df):
    print("=" * 90)
    print(f"Running first-improvement experiment: {source_name} -> {target_name}")
    print("=" * 90)
    print(
        f"source train/val = {len(source_train_df)}/{len(source_val_df)} | "
        f"target test = {len(target_test_df)}"
    )

    train_loader = make_loader(source_train_df, f"{source_name}_train", shuffle=True,  return_augmented=True)
    val_loader   = make_loader(source_val_df,   f"{source_name}_val",   shuffle=False, return_augmented=False)
    test_loader  = make_loader(target_test_df,  f"{target_name}_test",  shuffle=False, return_augmented=False)

    model = FirstImprovementModel(
        num_classes=cfg.NUM_CLASSES,
        embed_dim=cfg.EMBED_DIM,
        token_grid=cfg.TOKEN_GRID,
        n_heads=cfg.TRANSFORMER_HEADS,
        n_layers=cfg.TRANSFORMER_LAYERS,
        dropout=cfg.DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.AMP and DEVICE.type == "cuda"))

    best_ckpt_path = CHECKPOINT_ROOT / f"{source_name}_to_{target_name}__first_improvement_best.pt"
    best_val = float("-inf")
    history = []
    patience_counter = 0

    for epoch in range(1, cfg.EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer, scaler=scaler, train=True)
        val_metrics   = run_epoch(model, val_loader, optimizer=None, scaler=None, train=False)

        score = validation_score(val_metrics)
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_bacc": train_metrics["BACC"],
            "train_auc": train_metrics["AUC"],
            "val_loss": val_metrics["loss"],
            "val_bacc": val_metrics["BACC"],
            "val_auc": val_metrics["AUC"],
            "val_f1": val_metrics["F1"],
            "val_score": score,
        }
        history.append(row)

        improved = score > best_val
        if improved:
            best_val = score
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "best_val": best_val,
                "config": cfg.__dict__,
            }, best_ckpt_path)
        else:
            patience_counter += 1

        print(
            f"epoch {epoch:03d}/{cfg.EPOCHS} | "
            f"train_loss={train_metrics['loss']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_bacc={val_metrics['BACC']:.2f} | "
            f"val_auc={val_metrics['AUC']:.2f} | "
            f"best={best_val:.2f}"
        )

        if patience_counter >= cfg.EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])

    test_metrics = run_epoch(model, test_loader, optimizer=None, scaler=None, train=False)
    test_probs, test_labels, test_proto_dist, test_alpha = predict_loader(model, test_loader)
    test_preds = (test_probs >= 0.5).astype(int)
    cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

    history_df = pd.DataFrame(history)
    history_csv = WORK_ROOT / f"{source_name}_to_{target_name}__history.csv"
    history_df.to_csv(history_csv, index=False)

    summary = {
        "experiment": f"{source_name}_to_{target_name}",
        "best_val": float(best_val),
        "ood_test_metrics": {k: float(v) for k, v in test_metrics.items()},
        "mean_prototype_distance": float(test_proto_dist.mean()),
        "mean_harmonization_alpha": float(test_alpha.mean()),
        "confusion_matrix": cm.tolist(),
        "n_source_train": int(len(source_train_df)),
        "n_source_val": int(len(source_val_df)),
        "n_target_test": int(len(target_test_df)),
        "checkpoint": str(best_ckpt_path),
        "history_csv": str(history_csv),
    }

    summary_json = WORK_ROOT / f"{source_name}_to_{target_name}__summary.json"
    with open(summary_json, "w") as f:
        json.dump(summary, f, indent=2)

    print("\nOOD test metrics:")
    for k in ["ACC", "BACC", "SEN", "SPE", "AUC", "F1"]:
        print(f"{k}: {test_metrics[k]:.2f}")
    print("Mean prototype distance:", float(test_proto_dist.mean()))
    print("Mean harmonization alpha:", float(test_alpha.mean()))
    print("Confusion matrix:\n", cm)

    return {
        "summary": summary,
        "history": history_df,
        "test_probs": test_probs,
        "test_labels": test_labels,
        "test_proto_dist": test_proto_dist,
        "test_alpha": test_alpha,
    }

## 9) Run experiments

In [ ]:
directions = []

if cfg.RUN_BOTH_DIRECTIONS:
    directions = [
        ("ADNI1", "ADNI2", adni1_train, adni1_val, adni2_test),
        ("ADNI2", "ADNI1", adni2_train, adni2_val, adni1_test),
    ]
else:
    if cfg.RUN_DIRECTION == "ADNI1_TO_ADNI2":
        directions = [("ADNI1", "ADNI2", adni1_train, adni1_val, adni2_test)]
    elif cfg.RUN_DIRECTION == "ADNI2_TO_ADNI1":
        directions = [("ADNI2", "ADNI1", adni2_train, adni2_val, adni1_test)]
    else:
        raise ValueError(f"Unknown RUN_DIRECTION: {cfg.RUN_DIRECTION}")

all_results = {}
for source_name, target_name, source_train_df, source_val_df, target_test_df in directions:
    result = run_direction(source_name, target_name, source_train_df, source_val_df, target_test_df)
    all_results[f"{source_name}_to_{target_name}"] = result["summary"]

all_summary_path = WORK_ROOT / "first_improvement_all_results.json"
with open(all_summary_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("\nSaved combined summary to:", all_summary_path)
pd.DataFrame(all_results).T

Running first-improvement experiment: ADNI1 -> ADNI2
source train/val = 22/17 | target test = 73


  0%|          | 0/11 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

epoch 001/30 | train_loss=0.8547 | val_loss=0.6884 | val_bacc=50.00 | val_auc=58.57 | best=50.00


  0%|          | 0/11 [00:00<?, ?it/s]

...

  0%|          | 0/37 [00:00<?, ?it/s]

epoch 009/30 | train_loss=0.6584 | val_loss=0.6411 | val_bacc=50.00 | val_auc=52.71 | best=50.00
Early stopping at epoch 9


  0%|          | 0/9 [00:00<?, ?it/s]


OOD test metrics:
ACC: 55.56
BACC: 50.00
SEN: 0.00
SPE: 100.00
AUC: 25.00
F1: 0.00
Mean prototype distance: 2.5936195850372314
Mean harmonization alpha: 0.47961828112602234
Confusion matrix:
 [[10  0]
 [ 8  0]]

Saved combined summary to: /content/first_improvement_work/first_improvement_all_results.json


,experiment,best_val,ood_test_metrics,mean_prototype_distance,mean_harmonization_alpha,confusion_matrix,n_source_train,n_source_val,n_target_test,checkpoint,history_csv
ADNI1_to_ADNI2,ADNI1_to_ADNI2,50.0,"{'ACC': 65.75342465753424, 'BACC': 50.0, 'SEN'...",8.365376,0.226994,"[[48, 0], [25, 0]]",22,17,73,/content/first_improvement_work/checkpoints/AD...,/content/first_improvement_work/ADNI1_to_ADNI2...
ADNI2_to_ADNI1,ADNI2_to_ADNI1,50.0,"{'ACC': 55.55555555555556, 'BACC': 50.0, 'SEN'...",2.59362,0.479618,"[[10, 0], [8, 0]]",97,73,18,/content/first_improvement_work/checkpoints/AD...,/content/first_improvement_work/ADNI2_to_ADNI1...


## 10) Check cached sample


In [ ]:
sample_a = adni1_train.iloc[0].to_dict()
mri_a = get_cached_or_build(sample_a, modality="MRI", index_key="ADNI1_MRI")
pet_a = get_cached_or_build(sample_a, modality="PET", index_key="ADNI1_PET")
print("ADNI1 MRI shape:", mri_a.shape, "PET shape:", pet_a.shape)

sample_b = adni2_train.iloc[0].to_dict()
mri_b = get_cached_or_build(sample_b, modality="MRI", index_key="ADNI2_MRI")
pet_b = get_cached_or_build(sample_b, modality="PET", index_key="ADNI2_PET")
print("ADNI2 MRI shape:", mri_b.shape, "PET shape:", pet_b.shape)

ADNI1 MRI shape: (96, 112, 96) PET shape: (96, 112, 96)
ADNI2 MRI shape: (96, 112, 96) PET shape: (96, 112, 96)


NOTE: The accuracy in ADNI1 -> ADNI2 case is same as baseline while the accuracy in ADNI2 -> ADNI1 case is almost 22% greater. In the baseline model we were passing the testing data through the model without labels during training but in this model we are not showing the testing data to the model during training and it is giving better accuracy in one case than baseline and same in the other one. So thats the improvement and now in the second improvement we will try to increase the accuracy further and also try to test it on totally different dataset other than ADNI and see how it performs in that case. 